In [29]:
import pandas as pd
import asyncio
import re
from pathlib import Path
from playwright.async_api import async_playwright

# Membaca 50 kandidat
df_50 = pd.read_csv("data/kandidat_50_putusan.csv")

# Folder PDF
FOLDER_PDF = Path("PDF_Putusan")

for tahun in [2023, 2024, 2025]:
    (FOLDER_PDF / str(tahun)).mkdir(
        parents=True,
        exist_ok=True
    )

print("Total kandidat:", len(df_50))
print("Folder PDF siap digunakan.")

Total kandidat: 50
Folder PDF siap digunakan.


In [15]:
from pathlib import Path

# Folder penyimpanan PDF putusan
FOLDER_PDF = Path("PDF_Putusan")
FOLDER_PDF.mkdir(parents=True, exist_ok=True)

print("Folder PDF:", FOLDER_PDF.resolve())

Folder PDF: D:\UMM\SEMESTER 7\Temu Kembali INFORMASI\UTS Project\Crawler_Putusan_MA_FINAL\Crawler_Putusan_MA_FINAL\PDF_Putusan


In [16]:
def nama_file_aman(no_putusan):
    nama = re.sub(r"[^a-zA-Z0-9]+", "_", str(no_putusan))
    return f"putusan_{nama}.pdf"


async def download_putusan(browser, row):

    tahun = int(row["Tahun Filter"])
    no_putusan = str(row["No Putusan"])
    url_detail = row["URL Detail"]

    folder_tahun = FOLDER_PDF / str(tahun)
    folder_tahun.mkdir(parents=True, exist_ok=True)

    lokasi_file = folder_tahun / nama_file_aman(no_putusan)

    if lokasi_file.exists():
        with open(lokasi_file, "rb") as f:
            if f.read(5) == b"%PDF-":
                return {
                    "No Putusan": no_putusan,
                    "Tahun": tahun,
                    "Status": "BERHASIL_SUDAH_ADA",
                    "File": str(lokasi_file)
                }

    context = browser.contexts[0]
    page = await context.new_page()

    try:
        response = await page.goto(
            url_detail,
            wait_until="domcontentloaded",
            timeout=60000
        )

        if response is None or response.status != 200:
            return {
                "No Putusan": no_putusan,
                "Tahun": tahun,
                "Status": "GAGAL_AKSES",
                "File": None
            }

        await page.wait_for_timeout(2000)

        pdf_links = await page.locator(
            'a[href*="/pdf/"]'
        ).evaluate_all("""
            elements => elements.map(a => a.href)
        """)

        pdf_links = [
            url for url in pdf_links
            if "/zip/" not in url.lower()
        ]

        if not pdf_links:
            return {
                "No Putusan": no_putusan,
                "Tahun": tahun,
                "Status": "PDF_TIDAK_DITEMUKAN",
                "File": None
            }

        pdf_url = pdf_links[0]

        async with page.expect_download(timeout=60000) as info:
            await page.evaluate("""
                (url) => {
                    const link = [...document.querySelectorAll('a')]
                        .find(a => a.href === url);
                    if (link) link.click();
                }
            """, pdf_url)

        download = await info.value
        await download.save_as(str(lokasi_file))

        with open(lokasi_file, "rb") as f:
            header = f.read(5)

        ukuran = lokasi_file.stat().st_size

        if header == b"%PDF-" and ukuran > 0:
            status = "BERHASIL"
        else:
            lokasi_file.unlink(missing_ok=True)
            status = "PDF_TIDAK_VALID"

        return {
            "No Putusan": no_putusan,
            "Tahun": tahun,
            "Status": status,
            "File": str(lokasi_file) if status == "BERHASIL" else None
        }

    except Exception as e:
        lokasi_file.unlink(missing_ok=True)

        return {
            "No Putusan": no_putusan,
            "Tahun": tahun,
            "Status": "ERROR: " + str(e)[:150],
            "File": None
        }

    finally:
        await page.close()

In [3]:
async def jalankan_download():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil = []

        for i, row in df_50.head(5).iterrows():

            print(
                f"[{i+1}/5] Download:",
                row["No Putusan"]
            )

            result = await download_putusan(browser, row)
            hasil.append(result)

            print("Status:", result["Status"])

            # Jeda agar tidak membebani server
            await asyncio.sleep(3)

        df_hasil = pd.DataFrame(hasil)

        df_hasil.to_csv(
            "data/hasil_download_pdf.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print("\nREKAP DOWNLOAD:")
        print(df_hasil["Status"].value_counts())

        print(
            "\nPDF valid:",
            len(list(FOLDER_PDF.glob("*/*.pdf")))
        )


await jalankan_download()

[1/5] Download: 2031/Pid.Sus/2023/PN Sby
Status: BERHASIL
[2/5] Download: 1512/Pid.Sus/2023/PN Sby
Status: BERHASIL
[3/5] Download: 2260/Pid.Sus/2023/PN Sby
Status: BERHASIL
[4/5] Download: 2166/Pid.Sus/2023/PN Sby
Status: BERHASIL
[5/5] Download: 270/Pid.Sus/2023/PN Sby
Status: BERHASIL

REKAP DOWNLOAD:
Status
BERHASIL    5
Name: count, dtype: int64

PDF valid: 5


In [4]:
# Membaca kembali 50 kandidat
df_50 = pd.read_csv("data/kandidat_50_putusan.csv")

# Menyiapkan file pencatatan hasil
file_log = Path("data/hasil_download_pdf.csv")

if file_log.exists():
    df_hasil = pd.read_csv(file_log)
else:
    df_hasil = pd.DataFrame(
        columns=["No Putusan", "Tahun", "Status", "File"]
    )

async def download_sisa_putusan():

    async with async_playwright() as p:

        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil_baru = []

        for i, row in df_50.iterrows():

            tahun = int(row["Tahun Filter"])
            no_putusan = str(row["No Putusan"])

            # Periksa apakah PDF sudah ada dan valid
            file_pdf = (
                FOLDER_PDF /
                str(tahun) /
                nama_file_aman(no_putusan)
            )

            if file_pdf.exists():
                with open(file_pdf, "rb") as f:
                    valid = f.read(5) == b"%PDF-"

                if valid:
                    print(f"[{i+1}/50] SUDAH ADA: {no_putusan}")
                    continue

            print(f"\n[{i+1}/50] Download: {no_putusan}")

            result = await download_putusan(browser, row)
            hasil_baru.append(result)

            print("Status:", result["Status"])

            # Simpan progres setiap dokumen
            if hasil_baru:
                df_update = pd.concat(
                    [df_hasil, pd.DataFrame(hasil_baru)],
                    ignore_index=True
                )

                df_update.to_csv(
                    file_log,
                    index=False,
                    encoding="utf-8-sig"
                )

            # Beri jeda antarpermintaan
            await asyncio.sleep(5)

        await browser.close()

        print("\nPROSES DOWNLOAD SELESAI")

        jumlah_valid = 0

        for tahun in [2023, 2024, 2025]:
            folder = FOLDER_PDF / str(tahun)

            for file in folder.glob("*.pdf"):
                with open(file, "rb") as f:
                    if f.read(5) == b"%PDF-":
                        jumlah_valid += 1

        print("TOTAL PDF VALID:", jumlah_valid)

await download_sisa_putusan()

[1/50] SUDAH ADA: 2031/Pid.Sus/2023/PN Sby
[2/50] SUDAH ADA: 1512/Pid.Sus/2023/PN Sby
[3/50] SUDAH ADA: 2260/Pid.Sus/2023/PN Sby
[4/50] SUDAH ADA: 2166/Pid.Sus/2023/PN Sby
[5/50] SUDAH ADA: 270/Pid.Sus/2023/PN Sby

[6/50] Download: 1982/Pid.Sus/2023/PN Sby
Status: BERHASIL

[7/50] Download: 140/Pid.Sus/2023/PN Sby
Status: BERHASIL

[8/50] Download: 2258/Pid.Sus/2023/PN Sby
Status: BERHASIL

[9/50] Download: 2325/Pid.Sus/2023/PN Sby
Status: BERHASIL

[10/50] Download: 2043/Pid.Sus/2023/PN Sby
Status: BERHASIL

[11/50] Download: 2047/Pid.Sus/2023/PN Sby
Status: BERHASIL

[12/50] Download: 1880/Pid.Sus/2023/PN Sby
Status: BERHASIL

[13/50] Download: 1061 K/PDT.SUS-PHI/2026
Status: BERHASIL

[14/50] Download: 2189/Pid.Sus/2023/PN Sby
Status: BERHASIL

[15/50] Download: 1024/Pid.Sus/2023/PN Sby
Status: BERHASIL

[16/50] Download: 2274/Pid.Sus/2023/PN Sby
Status: BERHASIL

[17/50] Download: 2359/Pid.Sus/2023/PN Sby
Status: BERHASIL

[18/50] Download: 2054/Pid.Sus/2024/PN Sby
Status: BERHASIL

TAMBAHAN 2 FILE PENGGANTI

In [2]:
import pandas as pd

In [3]:
df_pengganti = pd.read_csv(
    "data/dua_kandidat_pengganti.csv"
)

print("Jumlah PDF yang akan diunduh:", len(df_pengganti))
display(df_pengganti[["No Putusan", "URL PDF"]])

Jumlah PDF yang akan diunduh: 2


,No Putusan,URL PDF
0,1948/Pid.Sus/2023/PN Sby,https://putusan3.mahkamahagung.go.id/direktori...
1,898/Pid.Sus/2024/PN Sby,https://putusan3.mahkamahagung.go.id/direktori...


In [12]:
from pathlib import Path

# Folder penyimpanan PDF putusan
FOLDER_PDF = Path("PDF_Putusan")
FOLDER_PDF.mkdir(parents=True, exist_ok=True)

print("Folder PDF:", FOLDER_PDF.resolve())

Folder PDF: D:\UMM\SEMESTER 7\Temu Kembali INFORMASI\UTS Project\Crawler_Putusan_MA_FINAL\Crawler_Putusan_MA_FINAL\PDF_Putusan


In [6]:
from pathlib import Path

# Cari folder PDF berdasarkan struktur project
folder_pdf = Path("PDF_Putusan")

for _, row in df_pengganti.iterrows():
    nomor = row["No Putusan"].split("/")[0]

    hasil = list(folder_pdf.rglob(f"*{nomor}*.pdf"))

    print(f"\n{row['No Putusan']}")
    if hasil:
        for file in hasil:
            print("PDF ditemukan:", file)
    else:
        print("PDF belum ditemukan")


1948/Pid.Sus/2023/PN Sby
PDF belum ditemukan

898/Pid.Sus/2024/PN Sby
PDF belum ditemukan


In [13]:
import asyncio
import pandas as pd
from pathlib import Path
from playwright.async_api import async_playwright

In [18]:
import re

In [19]:
async def download_dua_pengganti():

    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil = []

        for _, row in df_pengganti.iterrows():

            print("Mengunduh:", row["No Putusan"])

            result = await download_putusan(browser, row)
            hasil.append(result)

            print("Status:", result["Status"])

            await asyncio.sleep(3)

        df_hasil_pengganti = pd.DataFrame(hasil)

        df_hasil_pengganti.to_csv(
            "data/hasil_download_pengganti.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print("\nRekap:")
        print(df_hasil_pengganti["Status"].value_counts())


await download_dua_pengganti()

Mengunduh: 1948/Pid.Sus/2023/PN Sby
Status: BERHASIL
Mengunduh: 898/Pid.Sus/2024/PN Sby
Status: BERHASIL

Rekap:
Status
BERHASIL    2
Name: count, dtype: int64


In [21]:
df_hasil_pengganti = pd.read_csv(
    "data/hasil_download_pengganti.csv"
)

print("Hasil download dua pengganti:")
display(
    df_hasil_pengganti[
        ["No Putusan", "Status"]
    ]
)

Hasil download dua pengganti:


,No Putusan,Status
0,1948/Pid.Sus/2023/PN Sby,BERHASIL
1,898/Pid.Sus/2024/PN Sby,BERHASIL


In [23]:
print("KOLOM CSV LAMA:")
print(df_lama.columns.tolist())

print("\nKOLOM CSV PENGGANTI:")
print(df_baru.columns.tolist())

KOLOM CSV LAMA:
['No Putusan', 'Tahun', 'Status', 'File']

KOLOM CSV PENGGANTI:
['No Putusan', 'Tahun', 'Status', 'File']


In [24]:
# Baca ulang kedua CSV agar menggunakan data terbaru
df_lama = pd.read_csv("data/hasil_download_pdf.csv")
df_baru = pd.read_csv("data/hasil_download_pengganti.csv")

# Dua putusan anomali
anomali = [
    "1061 K/PDT.SUS-PHI/2026",
    "219 K/MIL/2026"
]

# Hapus dua anomali
df_lama = df_lama[
    ~df_lama["No Putusan"].isin(anomali)
].copy()

# Gabungkan 48 putusan lama dengan 2 pengganti
df_final = pd.concat(
    [df_lama, df_baru],
    ignore_index=True
)

# Simpan kembali CSV
df_final.to_csv(
    "data/hasil_download_pdf.csv",
    index=False,
    encoding="utf-8-sig"
)

# Periksa hasil
print("TOTAL PUTUSAN:", len(df_final))
print("\nDISTRIBUSI TAHUN:")
print(df_final["Tahun"].value_counts().sort_index())

display(df_final)

TOTAL PUTUSAN: 52

DISTRIBUSI TAHUN:
Tahun
2023    18
2024    18
2025    16
Name: count, dtype: int64


,No Putusan,Tahun,Status,File
0,2031/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2031_Pid_Sus_2023_PN_...
1,1512/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_1512_Pid_Sus_2023_PN_...
2,2260/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2260_Pid_Sus_2023_PN_...
3,2166/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2166_Pid_Sus_2023_PN_...
4,270/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_270_Pid_Sus_2023_PN_S...
5,1982/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_1982_Pid_Sus_2023_PN_...
6,140/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_140_Pid_Sus_2023_PN_S...
7,2258/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2258_Pid_Sus_2023_PN_...
8,2325/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2325_Pid_Sus_2023_PN_...
9,2043/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_2043_Pid_Sus_2023_PN_...


In [25]:
print(
    df_lama[
        df_lama["No Putusan"].astype(str).str.contains(
            "1061|219",
            case=False,
            na=False
        )
    ][["No Putusan", "Tahun", "Status", "File"]].to_string(index=False)
)

Empty DataFrame
Columns: [No Putusan, Tahun, Status, File]
Index: []


In [26]:
print("Jumlah baris:", len(df_final))
print("Jumlah nomor putusan unik:", df_final["No Putusan"].nunique())

print("\nNomor putusan yang duplikat:")
display(
    df_final[
        df_final["No Putusan"].duplicated(keep=False)
    ].sort_values("No Putusan")
)

Jumlah baris: 52
Jumlah nomor putusan unik: 50

Nomor putusan yang duplikat:


,No Putusan,Tahun,Status,File
48,1948/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_1948_Pid_Sus_2023_PN_...
50,1948/Pid.Sus/2023/PN Sby,2023,BERHASIL,PDF_Putusan\2023\putusan_1948_Pid_Sus_2023_PN_...
49,898/Pid.Sus/2024/PN Sby,2024,BERHASIL,PDF_Putusan\2024\putusan_898_Pid_Sus_2024_PN_S...
51,898/Pid.Sus/2024/PN Sby,2024,BERHASIL,PDF_Putusan\2024\putusan_898_Pid_Sus_2024_PN_S...


In [27]:
# Hapus duplikat, pertahankan satu baris untuk setiap nomor putusan
df_final = df_final.drop_duplicates(
    subset=["No Putusan"],
    keep="first"
).copy()

# Pastikan dua anomali lama sudah tidak ada
anomali = [
    "1061 K/PDT.SUS-PHI/2026",
    "219 K/MIL/2026"
]

df_final = df_final[
    ~df_final["No Putusan"].astype(str).str.strip().isin(anomali)
].copy()

# Simpan hasil akhir
df_final = df_final.reset_index(drop=True)

df_final.to_csv(
    "data/hasil_download_pdf.csv",
    index=False,
    encoding="utf-8-sig"
)

print("TOTAL PUTUSAN:", len(df_final))
print("\nDISTRIBUSI TAHUN:")
print(df_final["Tahun"].value_counts().sort_index())

print("\nNOMOR PUTUSAN UNIK:", df_final["No Putusan"].nunique())

TOTAL PUTUSAN: 50

DISTRIBUSI TAHUN:
Tahun
2023    17
2024    17
2025    16
Name: count, dtype: int64

NOMOR PUTUSAN UNIK: 50


In [28]:
from pathlib import Path

# Dua putusan anomali lama
anomali = [
    ("1061 K/PDT.SUS-PHI/2026", 2023),
    ("219 K/MIL/2026", 2024)
]

# Pastikan PDF pengganti tersedia
for _, row in df_baru.iterrows():
    file_baru = Path(row["File"])

    if not file_baru.exists():
        raise FileNotFoundError(
            f"PDF pengganti tidak ditemukan: {file_baru}"
        )

    print("PDF pengganti tersedia:", file_baru)

# Hapus hanya dua PDF anomali
for no_putusan, tahun in anomali:
    file_lama = (
        Path("PDF_Putusan")
        / str(tahun)
        / nama_file_aman(no_putusan)
    )

    if file_lama.exists():
        file_lama.unlink()
        print("Dihapus:", file_lama)
    else:
        print("File anomali tidak ditemukan:", file_lama)

# Hitung total PDF setelah pembaruan
semua_pdf = list(Path("PDF_Putusan").rglob("*.pdf"))

print("\nTOTAL PDF:", len(semua_pdf))

PDF pengganti tersedia: PDF_Putusan\2023\putusan_1948_Pid_Sus_2023_PN_Sby.pdf
PDF pengganti tersedia: PDF_Putusan\2024\putusan_898_Pid_Sus_2024_PN_Sby.pdf
Dihapus: PDF_Putusan\2023\putusan_1061_K_PDT_SUS_PHI_2026.pdf
Dihapus: PDF_Putusan\2024\putusan_219_K_MIL_2026.pdf

TOTAL PDF: 51


In [30]:
import pandas as pd
import asyncio
from playwright.async_api import async_playwright

# Ambil kandidat pengganti 2438
df_kandidat = pd.read_csv("data/kandidat_pdf.csv")

row_pengganti = df_kandidat[
    df_kandidat["No Putusan"].astype(str)
    == "2438/Pid.Sus/2023/PN Sby"
].iloc[0]

async def download_pengganti():
    async with async_playwright() as p:
        browser = await p.chromium.connect_over_cdp(
            "http://127.0.0.1:9222"
        )

        hasil = await download_putusan(browser, row_pengganti)

        print("\nHASIL DOWNLOAD:")
        print(hasil)

        await browser.close()

await download_pengganti()


HASIL DOWNLOAD:
{'No Putusan': '2438/Pid.Sus/2023/PN Sby', 'Tahun': 2023, 'Status': 'BERHASIL', 'File': 'PDF_Putusan\\2023\\putusan_2438_Pid_Sus_2023_PN_Sby.pdf'}


In [31]:
import fitz
import os

pdf_baru = "PDF_Putusan/2023/putusan_2438_Pid_Sus_2023_PN_Sby.pdf"

doc = fitz.open(pdf_baru)

print("Ukuran:", os.path.getsize(pdf_baru), "bytes")
print("Jumlah halaman:", len(doc))
print("Teks halaman pertama:")
print(doc[0].get_text()[:1000])

doc.close()

Ukuran: 252520 bytes
Jumlah halaman: 40
Teks halaman pertama:
hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
P U T U S A N
Nomor 2438/Pid.Sus/2023/PN Sby
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
Pengadilan Negeri Surabaya yang mengadili perkara pidana dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara Para Terdakwa :
Terdakwa 1
1. 
Nama lengkap 
:
I.N. DWI ANTANA DUSAK Anak Dari NYOMAN
2. 
Tempat lahir 
: 
Bogor
3. 
Umur/Tanggal lahir :
52 tahun /5 Juni 1971
4. 
Jenis kelamin 
:
Laki-laki
5. 
Kebangsaan 
:
Indonesia
6. 
Tempat tinggal 
:
Jl. Pakis Sidorejo III / 10 A RT 001 RW 005 Kel.
Pakis Kec. Sawahan Kota Surabaya
7. 
Agama 
:
Hindu
8. 
Pekerjaan 
:
Belum/tidak bekerja
Terdakwa 2
1. 
Nama lengkap 
:
ERWIN DWI KURNIAWAN Bin. SAMSURI
MARIMUN
2. 
Tempat lahir 
:
Kediri
3. 
Umur/Tanggal lahir 

d:\UMM\SEMESTER 7\Temu Kembali INFORMASI\UTS Project\Pengadilan_Negeri_Surabaya_Narkotika\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [32]:
import os

pdf_lama = "PDF_Putusan/2023/putusan_1948_Pid_Sus_2023_PN_Sby.pdf"
pdf_baru = "PDF_Putusan/2023/putusan_2438_Pid_Sus_2023_PN_Sby.pdf"

# Hapus PDF lama yang kosong
if os.path.exists(pdf_lama):
    os.remove(pdf_lama)

print("PDF 1948 kosong berhasil dihapus.")
print("PDF 2438 menjadi penggantinya.")

PDF 1948 kosong berhasil dihapus.
PDF 2438 menjadi penggantinya.
